# ⚡ FastAPI — APIs
## Python Ecosystem Tutorial Series — Module 5 of 18

**Author:** Himanshu Goel | [hgoelgithub.github.io](https://hgoelgithub.github.io)

---

| | |
|---|---|
| **Library** | ⚡ FastAPI |
| **Domain** | APIs |
| **Dataset** | Drug lookup REST API |
| **Module** | 5 of 18 |

**What you will learn:**

1. What FastAPI is and why it exists
2. Core concepts and data structures
3. Hands-on code with real data
4. Visualisations and interpretation
5. When to use it and alternatives

```bash
# Install required libraries
pip install fastapi
```

## Quick Reference Card

| Code | What it does |
|------|--------------|
| `@app.get("/path")` | Define GET route |
| `@app.post("/path")` | Define POST route |
| `BaseModel` | Validate request body |
| `HTTPException` | Return error codes |
| `/docs` | Auto-generated Swagger |

# 5. ⚡ FastAPI — APIs
> **Python + FastAPI = APIs**

FastAPI is the fastest way to build production-ready REST APIs in Python.
Auto-generates documentation, validates inputs, and is async-native.

**Key concepts:** routes, path parameters, query params, Pydantic models, async, auto-docs

In [ ]:
# ── FastAPI demo (shows the code structure; run with uvicorn in production) ───
# pip install fastapi uvicorn

print("""
┌─────────────────────────────────────────────────────────────────────┐
│                    FastAPI Drug Lookup API                          │
│                                                                     │
│  Endpoint: GET /drugs/{drug_name}                                   │
│  Endpoint: POST /predict-toxicity                                   │
│  Auto-docs: http://localhost:8000/docs                             │
└─────────────────────────────────────────────────────────────────────┘
""")

# This is the actual FastAPI application code:
fastapi_code = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional
import uvicorn

app = FastAPI(
    title="Drug Property API",
    description="Look up physicochemical properties of drugs",
    version="1.0.0"
)

# ── Database (in real life, this would be a SQL/MongoDB database) ─────────────
DRUG_DATABASE = {
    "aspirin":      {"mw": 180.16, "logp": 1.19, "hba": 4, "hbd": 2, "category": "NSAID"},
    "caffeine":     {"mw": 194.19, "logp": -0.07,"hba": 6, "hbd": 0, "category": "stimulant"},
    "ibuprofen":    {"mw": 206.28, "logp": 3.97, "hba": 2, "hbd": 1, "category": "NSAID"},
    "paracetamol":  {"mw": 151.16, "logp": 0.46, "hba": 3, "hbd": 2, "category": "analgesic"},
    "warfarin":     {"mw": 308.33, "logp": 2.70, "hba": 5, "hbd": 1, "category": "anticoagulant"},
}

# ── Pydantic model: validates request body automatically ─────────────────────
class ToxicityRequest(BaseModel):
    smiles: str
    molecular_weight: float
    logp: float
    name: Optional[str] = "Unknown"

class ToxicityResponse(BaseModel):
    name: str
    risk_level: str
    predicted_ld50: float
    confidence: float

# ── Route 1: GET a drug by name ───────────────────────────────────────────────
@app.get("/drugs/{drug_name}")
async def get_drug(drug_name: str, include_lipinski: bool = True):
    drug = DRUG_DATABASE.get(drug_name.lower())
    if not drug:
        raise HTTPException(status_code=404, detail=f"Drug {drug_name!r} not found")
    result = {"name": drug_name, **drug}
    if include_lipinski:
        # Lipinski Rule of 5 check for oral bioavailability
        result["lipinski_pass"] = (
            drug["mw"] <= 500 and drug["logp"] <= 5 and
            drug["hba"] <= 10 and drug["hbd"] <= 5
        )
    return result

# ── Route 2: GET all drugs (with optional category filter) ───────────────────
@app.get("/drugs")
async def list_drugs(category: Optional[str] = None):
    if category:
        return {k: v for k, v in DRUG_DATABASE.items() if v["category"] == category}
    return DRUG_DATABASE

# ── Route 3: POST predict toxicity ───────────────────────────────────────────
@app.post("/predict-toxicity", response_model=ToxicityResponse)
async def predict_toxicity(request: ToxicityRequest):
    # Simple rule-based prediction (replace with real ML model)
    if request.logp > 5 or request.molecular_weight > 500:
        risk, ld50 = "HIGH", 50.0
    elif request.logp > 3:
        risk, ld50 = "MODERATE", 500.0
    else:
        risk, ld50 = "LOW", 2000.0
    return ToxicityResponse(
        name=request.name, risk_level=risk,
        predicted_ld50=ld50, confidence=0.85
    )

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

print(fastapi_code)
print("\nTo run:")
print("  uvicorn main:app --reload")
print("  Open browser: http://localhost:8000/docs  (Swagger UI, auto-generated!)")
print("  Try: GET http://localhost:8000/drugs/aspirin")

In [ ]:
# ── Simulate API responses without a running server ──────────────────────────
import json

DRUG_DATABASE = {
    "aspirin":      {"mw": 180.16, "logp": 1.19, "hba": 4, "hbd": 2, "category": "NSAID"},
    "caffeine":     {"mw": 194.19, "logp":-0.07,  "hba": 6, "hbd": 0, "category": "stimulant"},
    "ibuprofen":    {"mw": 206.28, "logp": 3.97,  "hba": 2, "hbd": 1, "category": "NSAID"},
    "paracetamol":  {"mw": 151.16, "logp": 0.46,  "hba": 3, "hbd": 2, "category": "analgesic"},
    "warfarin":     {"mw": 308.33, "logp": 2.70,  "hba": 5, "hbd": 1, "category": "anticoagulant"},
}

print("── Simulated API responses ──")
for drug_name, props in DRUG_DATABASE.items():
    lip = (props["mw"]<=500 and props["logp"]<=5 and props["hba"]<=10 and props["hbd"]<=5)
    resp = {"name": drug_name, **props, "lipinski_pass": lip}
    print(f"GET /drugs/{drug_name}")
    print(f"  Response: {json.dumps(resp, indent=4)[:150]}")
    print()

# ── Visualise drug properties ─────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

drugs = list(DRUG_DATABASE.keys())
mws   = [DRUG_DATABASE[d]["mw"]  for d in drugs]
logps = [DRUG_DATABASE[d]["logp"] for d in drugs]

cols = ["#3498DB","#E74C3C","#27AE60","#F1C40F","#8E44AD"]
axes[0].bar(drugs, mws, color=cols, alpha=0.85, edgecolor="white")
axes[0].axhline(500, c="r", ls="--", lw=1.5, label="Lipinski MW≤500")
axes[0].set_title("Molecular Weight", fontweight="bold"); axes[0].legend()
axes[0].set_ylabel("MW (Da)"); axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(drugs, logps, color=cols, alpha=0.85, edgecolor="white")
axes[1].axhline(5, c="r", ls="--", lw=1.5, label="Lipinski logP≤5")
axes[1].set_title("logP (Lipophilicity)", fontweight="bold"); axes[1].legend()
axes[1].set_ylabel("logP"); axes[1].tick_params(axis="x", rotation=30)

plt.suptitle("FastAPI — Drug Database Properties", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fastapi_drugs.png", dpi=120, bbox_inches="tight")
plt.show()

## Deep Dive: FastAPI

### FastAPI vs Flask for APIs
FastAPI offers automatic input validation (Pydantic), auto-generated Swagger docs, native async support, and is one of the fastest Python web frameworks. Flask requires manual validation and documentation.

### Pydantic Models — Automatic Validation
```python
class ToxicityRequest(BaseModel):
    smiles: str         # MUST be a string
    mw: float           # MUST be a number
    name: str = "Unknown"  # optional, defaults to "Unknown"
```
If a client sends `mw: "hello"`, FastAPI automatically returns HTTP 422 Unprocessable Entity. No validation code needed.

### Three Parameter Types
```python
@app.get("/drugs/{name}")     # path param: part of the URL
@app.get("/drugs")             # query param: /drugs?category=NSAID
@app.post("/predict")          # request body: JSON sent in POST
```

### The Lipinski Rule of 5
Developed by Christopher Lipinski at Pfizer (1997). A drug is likely orally bioavailable if: MW <= 500 Da, logP <= 5, H-bond acceptors <= 10, H-bond donors <= 5.
Still the most widely used drug-likeness filter 27 years later.

### Auto-Documentation
Visit /docs after starting the server — full interactive Swagger UI generated automatically. You can test every endpoint in the browser with zero extra work.


## ✅ Key Takeaways — ⚡ FastAPI

1. FastAPI + Pydantic gives automatic input validation with zero extra code
2. Auto-generated /docs Swagger UI makes APIs self-documenting
3. Lipinski Rule of 5 remains the gold standard oral bioavailability filter
4. Async endpoints handle many concurrent requests efficiently

---
*Next: Continue to Module 6 of 18 in the Python Ecosystem Tutorial Series*  
*Portfolio: [hgoelgithub.github.io](https://hgoelgithub.github.io)*